# Flow Cartogram Timing Benchmarks

This page shows the results of benchmarking flow cartogram computations - both serial and parallel versions. Benchmarks were run on two datasets (Albers equal-area projection, metres):

- **US states** — 48 contiguous US states with Census population data.
- **Congressional districts** — ~430 contiguous US congressional districts
  (2020 Census, geometries simplified at 1 km tolerance).

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Locate benchmark_results.json relative to this notebook
_here = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
_json_path = os.path.join(_here, "benchmark_results.json")

if not os.path.exists(_json_path):
    # Also try the docs/explanations directory directly (when executed via nbconvert)
    _json_path = os.path.join(
        os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else ".",
        "docs",
        "explanations",
        "benchmark_results.json",
    )

_data = None
if os.path.exists(_json_path) and os.path.getsize(_json_path) > 0:
    try:
        with open(_json_path) as f:
            _data = json.load(f)
    except (json.JSONDecodeError, ValueError):
        pass

if _data is None:
    print(
        "benchmark_results.json not found or empty.\n"
        "Run the benchmarks first:\n"
        "  uv run pytest benchmarks/ --benchmark-enable "
        "--benchmark-json=docs/explanations/benchmark_results.json"
    )

In [ ]:
def _rounds_df(extra: dict) -> pd.DataFrame:
    """Expand extra_info into a per-round DataFrame.

    BenchmarkRuns.to_dict() produces a dict-of-lists (one entry per
    measured round).  Scalar-only extra_info (convergence tests) yields
    a single-row DataFrame.  Scalar keys that coexist with list keys
    are broadcast to all rows.
    """
    list_data = {k: v for k, v in extra.items() if isinstance(v, list)}
    scalar_data = {k: v for k, v in extra.items() if not isinstance(v, list)}
    if list_data:
        df = pd.DataFrame(list_data)
        for k, v in scalar_data.items():
            df[k] = v
    else:
        df = pd.DataFrame([scalar_data])
    return df


def _parse_fixed_iter_group(df):
    """Add variant, grid_size, per-round total_speedup, and per-phase speedup columns.

    Returns (df_per_round, phase_cols) where df_per_round has one row per
    benchmark round — suitable for seaborn plots that show error bars across
    rounds.  t_total_s is excluded from phase_cols (it is the sum, not a phase).
    """
    df = df.copy()
    df["config"] = df["name"].str.extract(r"\[([^\]]+)\]")
    df[["variant", "grid_size"]] = df["config"].str.rsplit("-", n=1, expand=True)
    df["grid_size"] = df["grid_size"].astype(int)

    # Per-round total-time speedup from t_total_s (one value per round)
    serial_total_means = df[df["variant"] == "serial"].groupby("grid_size")["t_total_s"].mean()
    df["total_speedup"] = df["grid_size"].map(serial_total_means) / df["t_total_s"]

    # Phase columns — t_total_s is the full-run sum, not a phase
    phase_cols = sorted(c for c in df.columns if c.startswith("t_") and c != "t_total_s")

    # Per-phase speedup: serial benchmark mean / this benchmark mean (per grid_size)
    for col in phase_cols:
        bm_phase_mean = df.groupby("name")[col].transform("mean")
        serial_means = df[df["variant"] == "serial"].groupby("grid_size")[col].mean()
        df[f"{col}_speedup"] = df["grid_size"].map(serial_means) / bm_phase_mean

    # fft_threads and density_threads are int columns from Benchmark.to_dict()

    return df, phase_cols


def _parse_vertex_group(df):
    """Parse Group D (test_vertex_scaling) names and compute speedups.

    Test IDs have the form [simplify_m-variant], e.g. [None-serial] or
    [1000-parallel (density)].  n_vertices (from Benchmark.to_dict()) is
    used as the x-axis in plots because it reflects actual computational
    load, not the simplification parameter.
    """
    df = df.copy()
    df["config"] = df["name"].str.extract(r"\[([^\]]+)\]")
    # simplify_m value is the first part; variant id (no hyphens) is the second
    df[["simplify_str", "variant"]] = df["config"].str.rsplit("-", n=1, expand=True)
    df["simplify_m"] = df["simplify_str"].apply(lambda x: None if x == "None" else int(x))

    # Per-round total-time speedup relative to serial at same simplify level
    serial_total_means = df[df["variant"] == "serial"].groupby("simplify_str")["t_total_s"].mean()
    df["total_speedup"] = df["simplify_str"].map(serial_total_means) / df["t_total_s"]

    phase_cols = sorted(c for c in df.columns if c.startswith("t_") and c != "t_total_s")
    return df, phase_cols

In [ ]:
if _data is not None:
    # collect all test data into a single dataframe
    all_rows = []
    for b in _data["benchmarks"]:
        rounds = _rounds_df(b.get("extra_info") or {})
        rounds["name"] = b["name"]
        rounds["mean_s"] = b["stats"]["mean"]  # benchmark-level stat, broadcast
        rounds["stddev_s"] = b["stats"]["stddev"]
        all_rows.append(rounds)
    df_all = pd.concat(all_rows, ignore_index=True)

    # extract grid size and parallelization tests for US States and Congressional Districts
    mask_a = df_all["name"].str.startswith("test_parallel_options")
    mask_b = df_all["name"].str.startswith("test_congressional_districts")

    df_a = df_all[mask_a | mask_b].copy()
    df_a.loc[df_a["name"].str.startswith("test_parallel_options"), "dataset"] = "US States"
    df_a.loc[df_a["name"].str.startswith("test_congressional_districts"), "dataset"] = "Congressional Districts"
    df_a, _phase_cols_a = _parse_fixed_iter_group(df_a)
    _phase_cols = _phase_cols_a

    # df_b = df_all[mask_b].copy()
    # df_b, _phase_cols_b = _parse_fixed_iter_group(df_b)

    # extract single- and multi-resolution convergence tests
    mask_c = df_all["name"].str.contains("convergence")
    df_c = df_all[mask_c].copy()
    df_c["algorithm"] = df_c["name"].str.replace("test_", "").str.replace("_convergence", "")

    # extract vertex count and parallelization tests
    mask_d = df_all["name"].str.startswith("test_vertex_scaling")
    df_d = df_all[mask_d].copy()
    df_d, _phase_cols_d = _parse_vertex_group(df_d)

The benchmarks were run on the following hardware and software:

In [ ]:
if _data is not None:
    from IPython.display import HTML

    mi = _data["machine_info"]
    ci = _data["commit_info"]
    cpu = mi.get("cpu", {})
    rows = [
        ("Run date", _data["datetime"][:10]),
        ("Host", mi.get("node", "—")),
        ("CPU", cpu.get("brand_raw", mi.get("processor", "—"))),
        ("Cores", str(cpu.get("count", "—"))),
        ("Python", mi.get("python_version", "—")),
        ("Commit", ci.get("id", "—")[:12] + (" (dirty)" if ci.get("dirty") else "")),
        ("Branch", ci.get("branch", "—")),
    ]
    html = (
        "<table style='border-collapse:collapse;font-size:0.85em'>"
        + "".join(
            f"<tr><td style='padding:2px 12px 2px 0;font-weight:bold'>{k}</td><td style='padding:2px 0'>{v}</td></tr>"
            for k, v in rows
        )
        + "</table>"
    )
    display(HTML(html))

## Grid size and parallelism comparisons

In these benchmark tests, the goal is to time the flow cartogram algorithm using various grid sizes (`grid_size` $\in$ { 256, 512, 1024, 2048 }) and parallelization options (`parallel_fft`, `parallel_density`).

The number of iterations is fixed (`n_iter=100`) to facilitate comparison. To make sure all tests run exactly 100 iterations, `mean_tol` and `max_tol` are set sufficiently low to prevent convergence, and early stopping is disabled (`stall_patience=None`). Speedups are computed relative to the serial baseline at the same `grid_size`.

Note that in practice, the actual time it takes to construct a flow cartogram will be determined by how many iterations are needed to converge to the desired tolerances.

In [ ]:
if _data is not None and not df_a.empty:
    display_df = (
        df_a.groupby(["dataset", "name"], sort=False)
        .agg(
            variant=("variant", "first"),
            grid_size=("grid_size", "first"),
            mean_s=("mean_s", "first"),
            stddev_s=("stddev_s", "first"),
            total_speedup=("total_speedup", "mean"),
            fft_threads=("fft_threads", "first"),
            density_threads=("density_threads", "first"),
        )
        .sort_values(["dataset", "grid_size", "variant"])
        .reset_index(drop=False)
        .drop(columns="name")
    )
    display_df["mean_s"] = display_df["mean_s"].map("{:.2f}".format)
    display_df["stddev_s"] = display_df["stddev_s"].map("{:.2f}".format)
    display_df["total_speedup"] = display_df["total_speedup"].map(lambda x: f"{x:.2f}x" if x == x else "")
    display_df.columns = [
        "Dataset",
        "Config",
        "Grid",
        "Mean (s)",
        "Std dev (s)",
        "Speedup vs serial",
        "FFT threads",
        "Density threads",
    ]
    display(display_df)

In [ ]:
_phase_order = ["setup", "density", "velocity", "displacement", "areas", "errors", "snapshot", "other"]

# Phases whose time should be divided by density_calls vs niterations
_DENSITY_PHASES = {"density", "velocity"}
_ITER_PHASES = {"displacement", "areas", "errors", "snapshot", "other"}

if _data is not None and not df_a.empty:
    df_a_melt = pd.melt(
        df_a,
        id_vars=["dataset", "variant", "grid_size", "round", "niterations", "density_calls"],
        value_vars=_phase_cols,
        var_name="phase",
        value_name="time_s",
    )
    df_a_melt["phase"] = df_a_melt["phase"].str[2:-2]  # t_setup_s → setup

    def _norm_ms(row):
        p = row["phase"]
        if p in _DENSITY_PHASES:
            return row["time_s"] / row["density_calls"] * 1000 if row["density_calls"] else float("nan")
        elif p in _ITER_PHASES:
            return row["time_s"] / row["niterations"] * 1000 if row["niterations"] else float("nan")
        else:  # setup — one-time cost, just convert to ms
            return row["time_s"] * 1000

    df_a_melt["time_ms"] = df_a_melt.apply(_norm_ms, axis=1)

In [ ]:
if _data is not None and not df_a.empty:
    # Compute per-round speedup: serial mean / each individual round's time_s.
    # Keeping one row per round lets seaborn show CIs across rounds in the speedup chart.
    serial_means_a = (
        df_a_melt[df_a_melt["variant"] == "serial"]
        .groupby(["dataset", "grid_size", "phase"])["time_s"]
        .mean()
        .rename("serial_mean_s")
    )
    va = df_a_melt.join(serial_means_a, on=["dataset", "grid_size", "phase"])
    va["speedup"] = va["serial_mean_s"] / va["time_s"]

In [ ]:
if _data is not None and not df_a.empty:
    display(va[["dataset", "variant", "grid_size", "phase", "round", "time_s", "time_ms", "speedup"]])

### Dataset information

In [ ]:
from IPython.display import Markdown

md = ""

for ds in ["US States", "Congressional Districts"]:
    niter, nupdates, ngeo, nvert = df_a[df_a["dataset"] == ds].iloc[0][
        ["niterations", "density_calls", "n_geometries", "n_vertices"]
    ]

    md += f"**{ds}**\n\n"
    md += f"Flow cartogram of **{int(ngeo)}** geometries with a total of **{int(nvert)}** vertices "
    md += f"({int(niter)} iterations completed, with {int(nupdates)} updates of the density and velocity fields).\n\n"

display(Markdown(md))

### Compute time scales with grid size

The `grid_size` parameter has a significant impact on runtime, especially with a large grid size of 2048 (left plot). One hundred iterations take close to 3 seconds for both datasets when using a grid size of 2048. Note that for both datasets, a grid size of 512 is generally sufficient to converge to reasonable tolerances.

The flow cartogram algorithm consists of a number of phases that the benchmark times separately. At a high level, there is initial setup, followed by *N* iterations during which density and velocity fields are computed (every *K*-th iteration as set by `recompute_every`), and vertices are displaced (every iteration). Here is a breakdown of the timed phases:

| Phase | What it measures |
|-------|-----------------|
| **setup** | Pre-loop: grid construction, FFTW plan, geometry unpacking. |
| **density** | `compute_density_field_from_geometries` — rasterises geometries onto the grid.  Only charged on iterations where the density is recomputed (`recompute_every`). |
| **velocity** | FFT Poisson solve (`pyfftw`) — same cadence as density. |
| **displacement** | `displace_coords_numba` — moves every coordinate by the velocity field.  Charged every iteration. |
| **areas** | `flat_geoms.compute_areas()` — recomputes polygon areas.  Charged every iteration. |
| **errors** | `compute_error_metrics()` — log-ratio error statistics.  Charged every iteration. |
| **snapshot** | Geometry reconstruction and snapshot object creation. Charged every iteration. |
| **other** | Overhead not attributed to the phases above (progress bar, convergence bookkeep

The two graphs on the right show how much time is spent in each of the phases for both datasets across the tested grid sizes. For small grid sizes, the largest contributors to compute time are the density field, displacement, and area phases. For large grid sizes, the largest contributors are the density field, velocity field, and other phases, which are the phases that are most impacted by increasing grid size.

In [ ]:
if _data is not None and not df_a.empty:
    grid_sizes = sorted(df_a["grid_size"].unique())

    fig, axes = plt.subplots(1, 3, figsize=(10, 4), gridspec_kw={"width_ratios": [1.5, 1, 1]}, sharey=True)

    # Left: wall time per round (seaborn computes mean + CI across rounds)
    _ = sns.lineplot(
        data=df_a[df_a["variant"] == "serial"],
        x="grid_size",
        y="t_total_s",
        ax=axes[0],
        hue="dataset",  # palette=palette,
        # hue="variant", hue_order = ['serial', 'parallel (velocity)', 'parallel (density)', 'parallel'], palette=palette,
    )
    # axes[0].set_xscale("log", base=2)
    axes[0].set_xticks(grid_sizes)
    axes[0].set_xticklabels(grid_sizes)
    axes[0].set_ylabel("Wall time (s)")
    axes[0].set_xlabel("grid size")
    axes[0].set_ylim(0, None)

    sns.move_legend(axes[0], "upper left", fontsize="x-small", title="")

    cmap = plt.cm.tab10
    phase_colors = {p: cmap(i) for i, p in enumerate(_phase_order)}

    for ax, ds in zip(axes[1:], ["US States", "Congressional Districts"], strict=False):
        serial_a = df_a_melt[(df_a_melt["dataset"] == ds) & (df_a_melt["variant"] == "serial")]
        # Mean time_ms per (grid_size, phase) across rounds
        pivot = (
            serial_a.groupby(["grid_size", "phase"])["time_s"]
            .mean()
            .unstack("phase")
            .reindex(columns=_phase_order, fill_value=0)
        )

        grid_sizes_v = pivot.index.tolist()
        xs = range(len(grid_sizes_v))
        bottom = [0.0] * len(grid_sizes_v)

        for phase in _phase_order:
            heights = pivot[phase].tolist()
            ax.bar(xs, heights, bottom=bottom, color=phase_colors[phase], label=phase)
            bottom = [b + h for b, h in zip(bottom, heights, strict=False)]

        ax.set_xticks(list(xs))
        ax.set_xticklabels(grid_sizes_v)
        ax.set_xlabel("grid size")
        # ax.set_ylabel("ms / compute")
        ax.set_title(ds, fontsize="small")
        ax.legend(loc="upper left", fontsize="x-small")

_ = fig.suptitle("Total and phase-specific compute times")

In [ ]:
if _data is not None and not df_a.empty:
    grid_sizes = sorted(df_a["grid_size"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(6, 4), sharey=True)

    cmap = plt.cm.tab10
    phase_colors = {p: cmap(i) for i, p in enumerate(_phase_order)}

    for ax, ds in zip(axes, ["US States", "Congressional Districts"], strict=False):
        serial_a = df_a_melt[(df_a_melt["dataset"] == ds) & (df_a_melt["variant"] == "serial")]
        # Mean time_ms per (grid_size, phase) across rounds
        pivot = (
            serial_a.groupby(["grid_size", "phase"])["time_ms"]
            .mean()
            .unstack("phase")
            .reindex(columns=_phase_order, fill_value=0)
        )

        grid_sizes_v = pivot.index.tolist()
        xs = range(len(grid_sizes_v))
        bottom = [0.0] * len(grid_sizes_v)

        for phase in _phase_order:
            heights = pivot[phase].tolist()
            ax.bar(xs, heights, bottom=bottom, color=phase_colors[phase], label=phase)
            bottom = [b + h for b, h in zip(bottom, heights, strict=False)]

        ax.set_xticks(list(xs))
        ax.set_xticklabels(grid_sizes_v)
        ax.set_xlabel("grid size")
        ax.set_title(ds, fontsize="small")
        ax.legend(loc="upper left", fontsize="x-small")

_ = axes[0].set_ylabel("ms / iteration or update")

_ = fig.suptitle("Phase-specific compute time per iteration or update")

### Parallelization may speed up computations depending on the grid size and the dataset

Parallelized versions of the density field and velocity field calculations can be used to further speed up the flow cartogram calculation. Two options control the use of parallelization: `parallel_fft` (for velocity field calculations) and `parallel_density` (for density field calculations). The table below shows the various combinations of these options that were tested in the benchmark (numbers indicate how many threads were used).

In [ ]:
md = "| variant | `parallel_fft` | `parallel_density` |\n"
md += "|---------| :------------: | :----------------: |\n"

for v in ["serial", "parallel (velocity)", "parallel (density)", "parallel"]:
    t1, t2 = df_a[df_a["variant"] == v].iloc[0][["fft_threads", "density_threads"]]
    md += f"| {v} | {int(t1)} | {int(t2)} |\n"

display(Markdown(md))

As shown in the graphs below, parallelization can provide a modest speedup (right plot), but only for larger grid sizes. At lower grid sizes, any potential speedup is negated by the additional overhead of parallelization. Note that for the Congressional Districts dataset, enabling parallelization of the density field calculations actually results in a marked slow down, especially at small grid sizes. The reason is that this datasets contains many small geometries (districts) with relatively few vertices each. The calculation per geometry is actually quite fast, which means that there is more time spend on paralleization overhead than the actual computation, resulting in a slow down.

In [ ]:
if _data is not None and not df_a.empty:
    palette = {
        "serial": "black",
        "parallel": "blueviolet",
        "parallel (velocity)": "thistle",
        "parallel (density)": "mediumpurple",
    }
    non_serial = df_a[df_a["variant"] != "serial"]

    fig, axes = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1, 1]}, sharey=True)

    for ax, ds in zip(axes, ["US States", "Congressional Districts"], strict=False):
        _ = sns.barplot(
            data=non_serial.query(f"dataset=='{ds}'"),
            x="grid_size",
            y="total_speedup",
            hue="variant",
            palette=palette,
            ax=ax,
            hue_order=["serial", "parallel (velocity)", "parallel (density)", "parallel"],
        )
        ax.axhline(1, color="k", zorder=0)
        ax.set_xlabel("grid size")
        ax.set_ylabel("speedup vs serial")
        ax.set_title(ds)

        sns.move_legend(ax, "upper left", fontsize="x-small", title="")

### Extra visualizations

In [ ]:
if _data is not None and not df_a.empty:
    # Per-round rows → seaborn shows bootstrap CIs across rounds automatically.
    # y=time_ms: density/velocity phases in ms/call; loop phases in ms/iteration; setup in ms.
    palette = {
        "serial": "black",
        "parallel": "blueviolet",
        "parallel (velocity)": "thistle",
        "parallel (density)": "mediumpurple",
    }

    # var, label = 'time_ms', 'ms / iteration or update'
    var, label = "time_s", "Wall time (s)"

    g = sns.catplot(
        data=df_a_melt,
        x="grid_size",
        y=var,
        hue="variant",
        col="phase",
        row="dataset",
        kind="bar",
        palette=palette,
        col_order=["setup", "density", "velocity"],  # ], _phase_order,
        hue_order=["serial", "parallel (velocity)", "parallel (density)", "parallel"],
    )
    for ax in g.axes.flat:
        ax.set_ylabel(label)

In [ ]:
if _data is not None and not df_a.empty:
    # va has one row per (variant, grid_size, phase, round) → CIs across parallel rounds
    # Facet by variant so all non-serial configs are shown
    variants_present = [
        v for v in ["parallel (velocity)", "parallel (density)", "parallel"] if v in va["variant"].unique()
    ]

    g = sns.catplot(
        data=va[va["variant"].isin(variants_present)],
        x="phase",
        y="speedup",
        hue="grid_size",
        order=_phase_order,
        kind="bar",
        col="variant",
        col_order=variants_present,
        row="dataset",
        height=4,
        aspect=1.4,
    )

    # g.add_legend(title="grid size")
    for ax in g.axes.flat:
        ax.axhline(1, color="black", zorder=0)
        ax.set_xlabel("phase")
        ax.set_ylabel("speedup vs serial")
        ax.tick_params(axis="x", rotation=30)

In [ ]:
if _data is not None and not df_a.empty:
    # Join t_total_s (excluded from phase_cols) back onto the per-round melt
    total_s_a = df_a[["dataset", "variant", "grid_size", "round", "t_total_s"]]
    pct_a = df_a_melt.merge(total_s_a, on=["dataset", "variant", "grid_size", "round"])
    pct_a["pct"] = pct_a["time_s"] / pct_a["t_total_s"] * 100

    # Mean % per (variant, grid_size, phase)
    pct_a_mean = pct_a.groupby(["dataset", "variant", "grid_size", "phase"])["pct"].mean().reset_index()

    cmap = plt.cm.tab10
    phase_colors = {p: cmap(i) for i, p in enumerate(_phase_order)}

    variants_a = [
        v
        for v in ["serial", "parallel (velocity)", "parallel (density)", "parallel"]
        if v in pct_a_mean["variant"].unique()
    ]

    fig, axes = plt.subplots(2, len(variants_a), figsize=(3.5 * len(variants_a), 8), sharey=True, sharex=True)
    if len(variants_a) == 1:
        axes = axes[:, None]

    for axrow, ds in zip(axes, ["US States", "Congressional Districts"], strict=False):
        for ax, variant in zip(axrow, variants_a, strict=False):
            sub = (
                pct_a_mean.query(f"dataset=='{ds}' and variant=='{variant}'")
                .pivot(index=["grid_size"], columns="phase", values="pct")
                .reindex(columns=_phase_order, fill_value=0)
            )

            grid_sizes_v = sub.index.tolist()
            xs = range(len(grid_sizes_v))
            bottom = [0.0] * len(grid_sizes_v)

            for phase in _phase_order:
                if phase not in sub.columns:
                    continue
                heights = sub[phase].tolist()
                ax.bar(xs, heights, bottom=bottom, color=phase_colors[phase])
                bottom = [b + h for b, h in zip(bottom, heights, strict=False)]

            ax.set_xticks(list(xs))
            ax.set_xticklabels(grid_sizes_v)
            ax.set_title(variant)
            ax.set_xlabel("grid size")
            if ax is axes[0]:
                ax.set_ylabel("% of total time")

    handles = [plt.Rectangle((0, 0), 1, 1, color=phase_colors[p]) for p in _phase_order]
    axes[0, -1].legend(handles, _phase_order, loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)
    fig.suptitle("Relative phase time breakdown — US states (top) and Congressional Districts (bottom)")
    fig.tight_layout()

## Comparison of single and multi-resolution morphs 

In this benchmark, single- and multi-resolution versions of flow cartogram creation are compared (using `morph_gdf` and `CartogramWorkflow.morph_multiresolution` respectively) on the US States dataset. In this test, `grid_size` is fixed to 512 for the single resolution case, and the `min_resolution` option for the multi-resolution case is set to 64 with `levels=4` such that the maximum resolution is also 512. The tolerance are set to `mean_tol=0.005` and `max_tol=0.01`, and the algorithms are run to convergence. Parallelization is not enabled.

The results in the table below shows that the multi-resolution version converges in fewer iterations and is about two times faster.

In [ ]:
if _data is not None and not df_c.empty:
    display_cols = ["algorithm", "mean_s", "stddev_s", "niterations", "status", "mean_error_pct", "max_error_pct"]
    display_df = df_c[display_cols].copy()
    display_df["mean_s"] = display_df["mean_s"].map("{:.2f}".format)
    display_df["stddev_s"] = display_df["stddev_s"].map("{:.2f}".format)
    display_df.columns = ["Algorithm", "Mean (s)", "Std dev (s)", "Iterations", "Status", "Mean error %", "Max error %"]
    display(display_df.reset_index(drop=True))

## Vertex scaling and parallelism

In these benchmark tests, the goal is to determine how the number of vertices in the input geometries impact the runtime of the flow cartogram algorithm. Using more or less simplified geometries from the US States dataset (simplification tolerance $\in$ {None, 100, 500, 1000, 5000, 10000 }), a range of vertex counts are generated. The `grid_size` option is fixed at 512.

Similar to the tests of the `grid_size` option, the number of iterations is fixed (`n_iter=100`) to facilitate comparison. To make sure all tests run exactly 100 iterations, `mean_tol` and `max_tol` are set sufficiently low to prevent convergence, and early stopping is disabled (`stall_patience=None`). Parallelization speedups are computed relative to the serial baseline at the same vertex count.

### Data preparation

In [ ]:
if _data is not None and not df_d.empty:
    df_d_melt = pd.melt(
        df_d,
        id_vars=["variant", "n_vertices", "simplify_str", "round", "niterations", "density_calls"],
        value_vars=_phase_cols_d,
        var_name="phase",
        value_name="time_s",
    )
    df_d_melt["phase"] = df_d_melt["phase"].str[2:-2]  # t_setup_s → setup
    df_d_melt["time_ms"] = df_d_melt.apply(_norm_ms, axis=1)

In [ ]:
if _data is not None and not df_d.empty:
    # Summary table — one row per (simplify level, variant)
    summary_d = (
        df_d.groupby(["simplify_str", "variant"], sort=False)
        .agg(
            simplify_m=("simplify_m", "first"),
            n_geometries=("n_geometries", "first"),
            n_vertices=("n_vertices", "first"),
            mean_s=("mean_s", "first"),
            stddev_s=("stddev_s", "first"),
            total_speedup=("total_speedup", "mean"),
        )
        .reset_index()
        .sort_values(["n_vertices", "variant"])
        .reset_index(drop=True)
    )
    summary_d["mean_s"] = summary_d["mean_s"].map("{:.2f}".format)
    summary_d["stddev_s"] = summary_d["stddev_s"].map("{:.2f}".format)
    summary_d["total_speedup"] = summary_d["total_speedup"].map(lambda x: f"{x:.2f}x" if pd.notna(x) else "—")
    display(
        summary_d.rename(
            columns={
                "simplify_m": "Simplify (m)",
                "n_geometries": "Geometries",
                "n_vertices": "Vertices",
                "mean_s": "Mean (s)",
                "stddev_s": "Std dev (s)",
                "total_speedup": "Speedup vs serial",
            }
        )
    )

In [ ]:
if _data is not None and not df_d.empty:
    # Compute per-round speedup: serial mean / each individual round's time_s.
    # Keeping one row per round lets seaborn show CIs across rounds in the speedup chart.
    serial_means_d = (
        df_d_melt[df_d_melt["variant"] == "serial"]
        .groupby(["n_vertices", "phase"])["time_s"]
        .mean()
        .rename("serial_mean_s")
    )
    vd = df_d_melt.join(serial_means_d, on=["n_vertices", "phase"])
    vd["speedup"] = vd["serial_mean_s"] / vd["time_s"]

In [ ]:
if _data is not None and not df_d.empty:
    display(vd[["variant", "n_vertices", "phase", "round", "time_s", "time_ms", "speedup"]])

### Compute time scales with the number of vertices in input geometries

The run time of flow cartogram construction grows close to linear with increasing number of vertices in the input geometries (left plot), at least over the range of vertex counts tested.

Inpection of the phase-specific time measurements shows that the density field, displacement, and area computations are most impacted by the vertex count (right plot), as expected.

In [ ]:
if _data is not None and not df_d.empty:
    nvert = sorted(df_d["n_vertices"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1, 1]}, sharey=True)

    # Left: wall time per round (seaborn computes mean + CI across rounds)
    _ = sns.lineplot(
        data=df_d[df_d["variant"] == "serial"],
        x="n_vertices",
        y="t_total_s",
        ax=axes[0],
        color="k",  # hue='dataset', # palette=palette,
        # hue="variant", hue_order = ['serial', 'parallel (velocity)', 'parallel (density)', 'parallel'], palette=palette,
    )
    # axes[0].set_xscale("log", base=2)
    # axes[0].set_xticks(nvert)
    # axes[0].set_xticklabels(nvert)
    axes[0].set_ylabel("Wall time (s)")
    axes[0].set_xlabel("number of vertices")
    axes[0].set_ylim(0, None)

    # sns.move_legend(axes[0], 'upper left', fontsize='x-small', title='')

    cmap = plt.cm.tab10
    phase_colors = {p: cmap(i) for i, p in enumerate(_phase_order)}

    serial_d = df_d_melt[df_d_melt["variant"] == "serial"]
    # Mean time_ms per (grid_size, phase) across rounds
    pivot = (
        serial_d.groupby(["n_vertices", "phase"])["time_s"]
        .mean()
        .unstack("phase")
        .reindex(columns=_phase_order, fill_value=0)
    )

    nvert_v = [int(v) for v in pivot.index.tolist()]
    xs = range(len(nvert_v))
    bottom = [0.0] * len(nvert_v)

    for phase in _phase_order:
        heights = pivot[phase].tolist()
        axes[1].bar(xs, heights, bottom=bottom, color=phase_colors[phase], label=phase)
        bottom = [b + h for b, h in zip(bottom, heights, strict=False)]

    axes[1].set_xticks(list(xs))
    axes[1].set_xticklabels(nvert_v, rotation=45, ha="right")
    axes[1].set_xlabel("number of vertices")
    # ax.set_ylabel("ms / compute")
    # axes[1].set_title(ds, fontsize='small')
    axes[1].legend(loc="upper left", fontsize="x-small")

_ = fig.suptitle("Total and phase-specific compute times")

In [ ]:
if _data is not None and not df_d.empty:
    nvert = sorted(df_d["n_vertices"].unique())

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))

    cmap = plt.cm.tab10
    phase_colors = {p: cmap(i) for i, p in enumerate(_phase_order)}

    serial_d = df_d_melt[df_d_melt["variant"] == "serial"]
    # Mean time_ms per (grid_size, phase) across rounds
    pivot = (
        serial_d.groupby(["n_vertices", "phase"])["time_ms"]
        .mean()
        .unstack("phase")
        .reindex(columns=_phase_order, fill_value=0)
    )

    nvert_v = [int(v) for v in pivot.index.tolist()]
    xs = range(len(nvert_v))
    bottom = [0.0] * len(nvert_v)

    for phase in _phase_order:
        heights = pivot[phase].tolist()
        ax.bar(xs, heights, bottom=bottom, color=phase_colors[phase], label=phase)
        bottom = [b + h for b, h in zip(bottom, heights, strict=False)]

    ax.set_xticks(list(xs))
    ax.set_xticklabels(nvert_v, rotation=45, ha="right")
    ax.set_xlabel("number of vertices")
    ax.legend(loc="lower center", fontsize="x-small", ncols=2)

    _ = ax.set_ylabel("ms / iteration or update")

    _ = fig.suptitle("Phase-specific compute time per iteration or update")

### Parallelization speeds up computations for high vertex counts

Turning on parallelization produces a modest speed up, but only for a high number of vertices. As expected, this speed up is only observed for `parallel_density` option, but not `parallel_fft` option, because the velocity field computute time is mainly dependent on the grid size and not the number of vertices. Note that while here we look at the total number of vertices in the input geometries, parallelization of the density field computation is performed at the level of geometries, and so the speed up is rather dependent on the number of vertices per geometry. Thus, a dataset like the COngressional Districts with many geometries with fewer vertices per geometry, may not so a similar speed up.

In [ ]:
if _data is not None and not df_d.empty:
    palette = {
        "serial": "black",
        "parallel": "blueviolet",
        "parallel (velocity)": "thistle",
        "parallel (density)": "mediumpurple",
    }
    non_serial = df_d[df_d["variant"] != "serial"]

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))

    _ = sns.barplot(
        data=non_serial,
        x="n_vertices",
        y="total_speedup",
        hue="variant",
        palette=palette,
        ax=ax,
        hue_order=["serial", "parallel (velocity)", "parallel (density)", "parallel"],
    )
    ax.axhline(1, color="k", zorder=0)
    ax.set_xlabel("number of vertices")
    ax.set_ylabel("speedup vs serial")

    ax.set_xticks(range(len(nvert_v)))
    ax.set_xticklabels(nvert_v, rotation=45, ha="right")

    sns.move_legend(ax, "lower center", fontsize="x-small", title="")

### Extra visualizations

In [ ]:
if _data is not None and not df_d.empty:
    # Per-round rows → seaborn shows bootstrap CIs across rounds automatically.
    # y=time_ms: density/velocity phases in ms/call; loop phases in ms/iteration; setup in ms.
    palette = {
        "serial": "black",
        "parallel": "blueviolet",
        "parallel (velocity)": "thistle",
        "parallel (density)": "mediumpurple",
    }

    # var, label = 'time_ms', 'ms / iteration or update'
    var, label = "time_s", "Wall time (s)"

    g = sns.catplot(
        data=df_d_melt,
        x="n_vertices",
        y=var,
        hue="variant",
        col="phase",
        kind="bar",
        palette=palette,
        col_order=["setup", "density", "velocity"],  # ], _phase_order,
        hue_order=["serial", "parallel (velocity)", "parallel (density)", "parallel"],
    )
    for ax in g.axes.flat:
        ax.set_ylabel(label)

In [ ]:
if _data is not None and not df_d.empty:
    # va has one row per (variant, grid_size, phase, round) → CIs across parallel rounds
    # Facet by variant so all non-serial configs are shown
    variants_present = [
        v for v in ["parallel (velocity)", "parallel (density)", "parallel"] if v in vd["variant"].unique()
    ]

    g = sns.catplot(
        data=vd[vd["variant"].isin(variants_present)],
        x="phase",
        y="speedup",
        hue="n_vertices",
        order=_phase_order,
        kind="bar",
        col="variant",
        col_order=variants_present,
        height=4,
        aspect=1.4,
    )

    # g.add_legend(title="grid size")
    for ax in g.axes.flat:
        ax.axhline(1, color="black", zorder=0)
        ax.set_xlabel("phase")
        ax.set_ylabel("speedup vs serial")
        ax.tick_params(axis="x", rotation=30)

In [ ]:
if _data is not None and not df_d.empty:
    # Join t_total_s (excluded from phase_cols) back onto the per-round melt
    total_s_d = df_d[["variant", "n_vertices", "round", "t_total_s"]]
    pct_d = df_d_melt.merge(total_s_d, on=["variant", "n_vertices", "round"])
    pct_d["pct"] = pct_d["time_s"] / pct_d["t_total_s"] * 100

    # Mean % per (variant, grid_size, phase)
    pct_d_mean = pct_d.groupby(["variant", "n_vertices", "phase"])["pct"].mean().reset_index()

    cmap = plt.cm.tab10
    phase_colors = {p: cmap(i) for i, p in enumerate(_phase_order)}

    variants_d = [
        v
        for v in ["serial", "parallel (velocity)", "parallel (density)", "parallel"]
        if v in pct_d_mean["variant"].unique()
    ]

    fig, axes = plt.subplots(1, len(variants_d), figsize=(3.5 * len(variants_d), 4), sharey=True, sharex=True)
    if len(variants_d) == 1:
        axes = [axes]

    for ax, variant in zip(axes, variants_d, strict=False):
        sub = (
            pct_d_mean.query(f"variant=='{variant}'")
            .pivot(index=["n_vertices"], columns="phase", values="pct")
            .reindex(columns=_phase_order, fill_value=0)
        )

        nvert_v = [int(v) for v in sub.index.tolist()]
        xs = range(len(nvert_v))
        bottom = [0.0] * len(nvert_v)

        for phase in _phase_order:
            if phase not in sub.columns:
                continue
            heights = sub[phase].tolist()
            ax.bar(xs, heights, bottom=bottom, color=phase_colors[phase])
            bottom = [b + h for b, h in zip(bottom, heights, strict=False)]

        ax.set_xticks(list(xs))
        ax.set_xticklabels(nvert_v, rotation=45, ha="right")
        ax.set_title(variant)
        ax.set_xlabel("number of vertices")
        if ax is axes[0]:
            ax.set_ylabel("% of total time")

    handles = [plt.Rectangle((0, 0), 1, 1, color=phase_colors[p]) for p in _phase_order]
    axes[-1].legend(handles, _phase_order, loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)
    fig.suptitle("Relative phase time breakdown")
    fig.tight_layout()